# math_agent iteration

Follows `003_math_agent.ipynb` findings: apparent catastrophic forgetting was a measurement artifact — each 'eval' was 8 rollouts of a single problem, different problem each step. Fixes applied in `math_agent.py` / this notebook:

- `correctness` now requires a valid submit (no free-text regex fallback).
- `beta=0.01` KL to base in `GRPOConfig`.
- `eval_task` passed to trainer with `eval_steps=10, eval_limit=32` — real cross-step signal.

GPU 0 = vLLM server. Train on GPU 1.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_HOME"] = "/opt/nvidia/hpc_sdk/Linux_aarch64/24.11/cuda/12.6"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

import httpx
import torch

print(f"torch: {torch.__version__} | cuda devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {free//1024**3}GB free / {total//1024**3}GB")

print(f"vLLM health: {httpx.get('http://localhost:8000/health/', timeout=5).json()}")

## Run 1 — fixes only

Same training shape as 003 (bs=8, num_gen=8 → 1 prompt/grad step), just the three fixes:
- tight correctness scorer (requires submit)
- beta=0.01 KL to base
- real eval_task with 32 fixed test problems every 10 steps

Expect: correctness stays near 0 early (valid_submit does the bootstrap work), KL regulariser stops the prose-drift. If the eval curve is flat at 0 after 50 steps the signal genuinely isn't there at this model/budget and we try widening prompts / longer training.

In [ ]:
from trl import GRPOConfig
from inspect_rl.example.math_agent import get_task
from inspect_rl.trainer import inspect_rl_train

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
task_train = get_task(split="train")
task_eval = get_task(split="test")

grpo_config = GRPOConfig(
    output_dir="outputs/math_agent_iter1",
    max_steps=50,
    per_device_train_batch_size=8,
    num_generations=8,
    max_completion_length=512,
    temperature=1.0,
    learning_rate=5e-6,
    warmup_steps=10,
    bf16=True,
    fp16=False,
    report_to="none",
    run_name="math_agent_iter1",
    save_steps=100,
    logging_steps=1,
    use_vllm=True,
    vllm_mode="server",
    vllm_server_host="localhost",
    vllm_server_port=8000,
    beta=0.01,
    reward_weights=[4.0, 0.2, -0.3],
)
print(f"Training {MODEL} for {grpo_config.max_steps} steps…")
trainer = inspect_rl_train(
    task=task_train,
    model=MODEL,
    grpo_config=grpo_config,
    vllm_base_url="http://localhost:8000",
    dataset_limit=500,
    eval_task=task_eval,
    eval_steps=10,
    eval_limit=32,
)
print("Training complete.")

## Analyse

In [ ]:
from pathlib import Path
from inspect_ai.log import read_eval_log
from inspect_ai.model import ChatMessageAssistant

def summarise(lp):
    log = read_eval_log(str(lp))
    if log.status != "success":
        return f"status={log.status} n={len(log.samples or [])}"
    samples = log.samples or []
    n = len(samples)
    correct = sum(1 for s in samples if s.scores and s.scores.get("correctness") and s.scores["correctness"].value == 1.0)
    valid = sum(1 for s in samples if s.scores and s.scores.get("valid_submit") and s.scores["valid_submit"].value == 1.0)
    fails = sum(1 for s in samples if s.scores and s.scores.get("tool_call_failures") and s.scores["tool_call_failures"].value == 1.0)
    avg_chars = sum(sum(len(m.text or "") for m in s.messages if isinstance(m, ChatMessageAssistant)) for s in samples) / max(n, 1)
    return f"n={n:2d} correct={correct:2d}/{n} valid_submit={valid:2d}/{n} tool_fails={fails:2d}/{n} avg_chars={avg_chars:5.0f}"

for label, root in [("iter1", "math_agent_iter1"), ("iter4", "math_agent_iter4")]:
    root_path = Path(f"<redacted-home>/src/inspect-rl/outputs/{root}")
    if not root_path.exists():
        continue
    run_dir = sorted(root_path.glob("*"))[-1]
    val_logs = sorted((run_dir / "val_eval_logs").glob("*.eval"))
    print(f"\n=== {label}: {run_dir.name} — val_eval (32 test) — {len(val_logs)} evals ===")
    for i, lp in enumerate(val_logs):
        print(f"  eval {i}: {summarise(lp)}")

    per_step = sorted((run_dir / "eval_logs").glob("*.eval"))
    print(f"  [{len(per_step)} per-step rollouts]")
